# E — Baselines (GPU)

Without a baseline, "our verifier is 78% precise" answers nothing. This runs two
comparisons over **exactly the claims notebook D judged**, in **exactly D's label space**,
so all three systems can be scored against one set of human labels.

| System | What it knows |
|---|---|
| **D** — graph-grounded | the curriculum KG |
| **E1** — LLM-as-judge, closed book | only its own parameters |
| **E2** — LLM-as-judge + retrieval | the top-3 textbook passages for the claim |

**E1 is the BenHalluEval analogue** — an LLM scoring claims from parametric knowledge.
It is the comparison the whole positioning rests on, since the argument is that a
structural check beats an opinion.

**E2 is the harder baseline and the more honest one.** A reviewer will ask whether the
graph earns its keep over plain retrieval from the same textbook. If E2 matches D, the KG
is expensive machinery for no gain, and better to find that out here than in review.

The judge is Qwen2.5-7B — the strongest model available, which makes the baselines as
hard to beat as they can be.

**Settings:** GPU `NvidiaTeslaT4`, Internet on, dataset `bangla-biology-kg`,
model `qwen-lm/qwen2.5` `7b-instruct`.

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

import importlib.util
assert importlib.util.find_spec("bitsandbytes"), "bitsandbytes missing — restart the session"
print("bitsandbytes ready")

In [ ]:
import json, glob, re, os, time
from pathlib import Path
import pandas as pd

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- config -------------------------------------------------------------
JUDGE   = "7b-instruct"
TOP_K   = 3        # passages given to E2
MAX_NEW = 96       # a label plus a short reason
BATCH   = 8
LIMIT   = None     # claims to judge; None = all
# -------------------------------------------------------------------------

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("eval")
WORK.mkdir(parents=True, exist_ok=True)
JUDGE_JL = WORK / "baseline_judgements.jsonl"


def find(name, *fallbacks):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for f in fallbacks:
        hits += glob.glob(f)
    if not hits:
        raise SystemExit(f"{name} not found — attach the dataset")
    return hits[0]


# D's verdict column is deliberately dropped: the baselines must not see it.
C = pd.read_csv(find("claim_verdicts.csv", "eval/claim_verdicts.csv",
                     "../eval/claim_verdicts.csv"))
C = C[["model", "qid", "claim_no", "chapter_no", "claim"]].drop_duplicates()
if LIMIT:
    C = C.head(LIMIT)

CORPUS = pd.read_parquet(find("biology_clean.parquet", "eval/biology_clean.parquet",
                              "../eval/biology_clean.parquet"))
print(f"{len(C)} claims to judge, {len(CORPUS)} textbook passages for retrieval")

## Retrieval for E2

TF-IDF over the same Biology passages the KG was built from, so E2 and D draw on identical
source material and differ only in representation — one keeps it as prose, the other as a
graph. Word-level analysis works for Bangla here because the corpus is whitespace-separated.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vec = TfidfVectorizer(analyzer="word", token_pattern=r"[ঀ-৿]+|[A-Za-z]+",
                      min_df=2, sublinear_tf=True)
M = vec.fit_transform(CORPUS.text.astype(str))
print(f"tf-idf matrix: {M.shape[0]} passages x {M.shape[1]} terms")

Qm = vec.transform(C.claim.astype(str))
sims = linear_kernel(Qm, M)
top = sims.argsort(axis=1)[:, ::-1][:, :TOP_K]

C = C.reset_index(drop=True)
C["passages"] = [[CORPUS.text.iloc[j] for j in row] for row in top]
C["top_sim"] = [round(float(sims[i, row[0]]), 3) for i, row in enumerate(top)]
print(f"median top-1 similarity: {C.top_sim.median():.3f}")
print(f"claims with no lexical overlap at all: {(C.top_sim == 0).sum()}")

## Judge prompts

Both baselines answer in D's label space. The instruction is deliberately about the
*curriculum*, not about truth in general — a claim can be true and still absent from a
Class 9–10 syllabus, and conflating those is the distinction the paper turns on.

In [ ]:
LABELS = {"সমর্থিত": "supported",
          "বিরোধী": "contradicted",
          "পাঠ্যক্রমে_নেই": "not_in_curriculum"}

SYSTEM = ("তুমি বাংলাদেশের নবম-দশম শ্রেণির জীববিজ্ঞান পাঠ্যবইয়ের একজন বিশেষজ্ঞ। "
          "তুমি শুধুমাত্র বৈধ JSON উত্তর দাও।")

RULES = """নিচের দাবিটি নবম-দশম শ্রেণির জীববিজ্ঞান পাঠ্যক্রম অনুযায়ী যাচাই করো।

তিনটি সম্ভাব্য রায়:
- "সমর্থিত": দাবিটি পাঠ্যক্রমের তথ্যের সাথে মিলে যায়।
- "বিরোধী": দাবিটি পাঠ্যক্রমের তথ্যের বিপরীত।
- "পাঠ্যক্রমে_নেই": দাবিটি সত্য হতে পারে, কিন্তু এই শ্রেণির পাঠ্যক্রমের অন্তর্ভুক্ত নয়।

শুধু এই ফরম্যাটে JSON দাও:
{{"রায়": "...", "কারণ": "এক বাক্যে"}}"""

CLOSED = RULES + "\n\nদাবি: {claim}"

RETRIEVED = (RULES + "\n\nপাঠ্যবই থেকে প্রাসঙ্গিক অংশ:\n{passages}\n\nদাবি: {claim}")


def build(tok, claim, passages=None):
    if passages is None:
        user = CLOSED.format(claim=claim)
    else:
        joined = "\n\n".join(f"[{i+1}] {p[:600]}" for i, p in enumerate(passages))
        user = RETRIEVED.format(passages=joined, claim=claim)
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)


def parse(text):
    """Recover the label even when the model wraps its JSON in prose or fences."""
    t = re.sub(r"^```(?:json)?|```$", "", str(text).strip(), flags=re.M)
    m = re.search(r"\{.*?\}", t, re.S)
    if m:
        try:
            obj = json.loads(m.group())
            lab = str(obj.get("রায়", "")).strip().strip('"')
            if lab in LABELS:
                return LABELS[lab], str(obj.get("কারণ", ""))[:200]
        except json.JSONDecodeError:
            pass
    # Fall back to a bare label mention rather than discarding the generation.
    for bn, en in LABELS.items():
        if bn in t:
            return en, ""
    return None, ""

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

cfgs = glob.glob(f"/kaggle/input/**/{JUDGE}/**/config.json", recursive=True)
if not cfgs:
    raise SystemExit(f"{JUDGE} not attached")
MODEL_PATH = str(Path(cfgs[0]).parent)
assert f"/{JUDGE}/" in MODEL_PATH + "/", f"resolved wrong model: {MODEL_PATH}"

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True),
    device_map="auto",
).eval()
print("judge loaded:", MODEL_PATH)

In [ ]:
done = set()
if JUDGE_JL.exists():
    with open(JUDGE_JL, encoding="utf-8") as f:
        done = {(r["system"], r["model"], r["qid"], r["claim_no"])
                for r in map(json.loads, filter(str.strip, f))}
    print(f"resuming — {len(done)} judgements already made")


@torch.inference_mode()
def run(prompts):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True)
            for o in out]


t_all = time.time()
with open(JUDGE_JL, "a", encoding="utf-8") as sink:
    for system, use_passages in [("E1_closed_book", False), ("E2_retrieval", True)]:
        todo = C[[(system, m, q, n) not in done
                  for m, q, n in zip(C.model, C.qid, C.claim_no)]].reset_index(drop=True)
        if not len(todo):
            print(f"{system}: already complete")
            continue

        print(f"\n{system}: {len(todo)} claims")
        t0 = time.time()
        for i in range(0, len(todo), BATCH):
            part = todo.iloc[i:i + BATCH]
            prompts = [build(tok, r.claim, r.passages if use_passages else None)
                       for r in part.itertuples()]
            try:
                raws = run(prompts)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print("\n  OOM — retrying one at a time")
                raws = [run([p])[0] for p in prompts]
            for r, raw in zip(part.itertuples(), raws):
                verdict, reason = parse(raw)
                sink.write(json.dumps({
                    "system": system, "model": r.model, "qid": r.qid,
                    "claim_no": int(r.claim_no), "claim": r.claim,
                    "verdict": verdict, "reason": reason, "raw": raw[:300],
                }, ensure_ascii=False) + "\n")
            sink.flush()
            n = min(i + BATCH, len(todo))
            el = time.time() - t0
            print(f"  {n}/{len(todo)}  {el:.0f}s  ({el/max(n,1):.2f}s/claim)", end="\r")
        print(f"\n  {system} done in {time.time()-t0:.0f}s")

print(f"\nboth baselines done in {time.time()-t_all:.0f}s")

## Compare the three systems

In [ ]:
B = pd.DataFrame([json.loads(l) for l in open(JUDGE_JL, encoding="utf-8") if l.strip()])
B = B.drop_duplicates(subset=["system", "model", "qid", "claim_no"])
unparsed = B.verdict.isna().sum()
print(f"{len(B)} judgements, {unparsed} unparseable ({unparsed/len(B)*100:.1f}%)")

D = pd.read_csv(find("claim_verdicts.csv", "eval/claim_verdicts.csv",
                     "../eval/claim_verdicts.csv"))
D_ = D[["model", "qid", "claim_no", "claim", "verdict"]].assign(system="D_graph")
ALL = pd.concat([D_, B[["model", "qid", "claim_no", "claim", "verdict", "system"]]],
                ignore_index=True)
ALL.to_csv(WORK / "all_system_verdicts.csv", index=False)

print("\nverdict distribution by system (%):")
print(pd.crosstab(ALL.system, ALL.verdict, normalize="index").mul(100).round(1).to_string())

wide = ALL.pivot_table(index=["model", "qid", "claim_no"], columns="system",
                       values="verdict", aggfunc="first")
wide = wide.dropna()
print(f"\nclaims judged by all three systems: {len(wide)}")
for a, b in [("D_graph", "E1_closed_book"), ("D_graph", "E2_retrieval"),
             ("E1_closed_book", "E2_retrieval")]:
    if a in wide.columns and b in wide.columns:
        print(f"  {a} vs {b}: agree on {(wide[a] == wide[b]).mean()*100:.1f}%")

print("\nNote: agreement is not accuracy. Which system is RIGHT needs the human labels")
print("from verdict_review_sample.csv — that is the next step, not this notebook.")